# Run E-Commerce Agent Pipeline on Kaggle (Local GPU)
This notebook uses **vLLM** to split an LLM across 2x T4 GPUs via Tensor Parallelism and run the multi-agent `main.py` pipeline locally without any API rate limits.

In [ ]:
# 1. Install required packages
!pip install -r requirements.txt
!pip install vllm nest_asyncio

In [ ]:
import subprocess
import time

# 2. Start vLLM OpenAI-Compatible Server in the background
# You can change this to any model on HuggingFace that fits in 30GB VRAM (e.g. Qwen/Qwen2.5-7B-Instruct, google/gemma-2-9b-it)
serve_cmd = """
python -m vllm.entrypoints.openai.api_server \
    --model kanocz/Qwen3.5-9B-Claude-4.6-HighIQ-THINKING-HERETIC-UNCENSORED-FP8-vLLM \
    --tensor-parallel-size 2 \
    --max-model-len 8192 \
    --dtype half \
    --port 8000
"""

print("Starting vLLM server across 2 GPUs...")
process = subprocess.Popen(serve_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# Wait for weights to load across GPUs
time.sleep(60)
print("Server should be ready at http://localhost:8000/v1")

In [ ]:
# 3. Run the LangGraph Multi-Agent Pipeline!
# Make sure tools/llm_client.py is configured to point to base_url="http://localhost:8000/v1"
!python main.py